# NLST cohort build — download, convert, discard

Fetches one CT series per exam for the 200-participant cohort, converts the slices
we keep into 256x256 PNGs on Drive, and deletes the DICOM immediately.

Resumable. Re-running skips any participant already recorded in the manifest,
so Colab disconnects cost at most one participant.

**Before running:** upload `cohort_nlst_200.csv` (folder icon in the left sidebar).

## 1. Setup

In [ ]:
!pip install -q pydicom

import os, io, gc, glob, json, shutil, zipfile, time
import requests, numpy as np, pandas as pd, pydicom
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

API      = "https://nlst.cancerimagingarchive.net/nbia-api/services/v1"
OUT      = "/content/drive/MyDrive/leakage_bench/nlst"      # PNGs + manifest live here
TMP      = "/content/tmp_dicom"                              # scratch, wiped constantly
MANIFEST = os.path.join(OUT, "manifest_nlst.csv")
STATUS   = os.path.join(OUT, "participant_status.csv")

os.makedirs(os.path.join(OUT, "images"), exist_ok=True)
os.makedirs(TMP, exist_ok=True)

# --- preprocessing, fixed for the whole study -------------------------------
SIZE          = 256      # 256x256, matching paper 1
SLICES_PER    = 10       # slices kept per exam
MIDDLE_FRAC   = 0.70     # sample from the middle 70%, skipping apex and diaphragm
MIN_SLICES    = 100      # a reconstruction must have at least this many
WINDOW_LEVEL  = -600     # standard lung window
WINDOW_WIDTH  = 1500

cohort = pd.read_csv("cohort_nlst_200.csv")
print(cohort.role.value_counts().to_string())
print("primary participants:", cohort[cohort.role=='primary'].pid.nunique())

## 2. The pipeline

`pick_series` is the series-selection rule: an `OPA` acquisition (not an `OPL`
scout) with at least 100 slices, and among those the **smallest**. Thin 300-slice
reconstructions cost double the transfer and buy nothing, since labels are per exam.

In [ ]:
def get_series(pid):
    r = requests.get(f"{API}/getSeries",
                     params={"Collection": "NLST", "PatientID": str(pid)}, timeout=90)
    r.raise_for_status()
    return r.json()

def pick_series(all_series, study_yr):
    cands = []
    for s in all_series:
        d = s.get("SeriesDescription", "")
        parts = d.split(",")
        if not parts or not parts[0].strip().isdigit():
            continue
        if int(parts[0]) != study_yr:
            continue
        if ",OPA," not in d:
            continue
        n = int(s.get("ImageCount", 0))
        if n < MIN_SLICES:
            continue
        cands.append((n, s))
    if not cands:
        return None
    cands.sort(key=lambda t: t[0])          # smallest adequate reconstruction
    return cands[0][1]

def download(uid, dest):
    r = requests.get(f"{API}/getImage", params={"SeriesInstanceUID": uid},
                     stream=True, timeout=1800)
    r.raise_for_status()
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(dest)

def ordered_slices(d):
    """Sort superior -> inferior by ImagePositionPatient z. Never InstanceNumber:
    it runs in opposite directions on different scanners."""
    out = []
    for p in glob.glob(os.path.join(d, "*.dcm")):
        try:
            ds = pydicom.dcmread(p)
        except Exception:
            continue
        if getattr(ds, "Modality", "") != "CT":
            continue
        if hasattr(ds, "ImagePositionPatient"):
            z = float(ds.ImagePositionPatient[2])
        elif hasattr(ds, "SliceLocation"):
            z = float(ds.SliceLocation)
        else:
            continue
        out.append((z, ds))
    out.sort(key=lambda t: -t[0])
    return [o[1] for o in out]

def to_png(ds):
    a  = ds.pixel_array.astype(np.float32)
    hu = a * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
    lo = WINDOW_LEVEL - WINDOW_WIDTH / 2
    hi = WINDOW_LEVEL + WINDOW_WIDTH / 2
    x  = np.clip((hu - lo) / (hi - lo), 0, 1)
    img = Image.fromarray((x * 255).astype(np.uint8))
    return img.resize((SIZE, SIZE), Image.BILINEAR)

def chosen_indices(n):
    lo = int(n * (1 - MIDDLE_FRAC) / 2)
    hi = n - lo - 1
    return [int(round(i)) for i in np.linspace(lo, hi, SLICES_PER)]

def append(path, rows):
    df = pd.DataFrame(rows)
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)

## 3. Run it

Leave running. Re-run the cell after any disconnect — it picks up where it stopped.
Reserves are used automatically if a primary participant turns out to have no images.

In [ ]:
done = set()
if os.path.exists(STATUS):
    done = set(pd.read_csv(STATUS).pid.astype(int))
    print("already processed:", len(done))

# primaries first, then reserves in rank order, stratum by stratum
order = (cohort[["pid","stratum","role","rank"]].drop_duplicates()
         .sort_values(["stratum","role","rank"], ascending=[True, True, True]))
target_per = cohort[cohort.role=="primary"].groupby("stratum").pid.nunique().to_dict()

ok_counts = {}
if os.path.exists(STATUS):
    st = pd.read_csv(STATUS)
    ok_counts = st[st.status=="ok"].groupby("stratum").pid.nunique().to_dict()

for _, prow in order.iterrows():
    pid, stratum = int(prow.pid), prow.stratum
    if pid in done:
        continue
    if ok_counts.get(stratum, 0) >= target_per[stratum]:
        continue                                   # stratum already full

    exams = cohort[(cohort.pid == pid)][["study_yr","label","stratum"]].drop_duplicates()
    t0 = time.time()
    try:
        allser = get_series(pid)
    except Exception as e:
        append(STATUS, [dict(pid=pid, stratum=stratum, status="api_error", note=str(e)[:120])])
        print(f"{pid}: api error"); continue

    if not allser:
        append(STATUS, [dict(pid=pid, stratum=stratum, status="no_images", note="")])
        print(f"{pid}: no images on TCIA -- using a reserve"); done.add(pid); continue

    rows, kept_exams = [], 0
    for _, e in exams.iterrows():
        s = pick_series(allser, int(e.study_yr))
        if s is None:
            continue
        dest = os.path.join(TMP, str(pid), str(e.study_yr))
        try:
            download(s["SeriesInstanceUID"], dest)
            sl = ordered_slices(dest)
            if len(sl) < MIN_SLICES:
                shutil.rmtree(dest, ignore_errors=True); continue
            for k, idx in enumerate(chosen_indices(len(sl))):
                ds = sl[idx]
                name = f"{pid}_T{int(e.study_yr)}_{k:02d}.png"
                rel  = os.path.join("images", name)
                to_png(ds).save(os.path.join(OUT, rel))
                rows.append(dict(
                    image_id=name[:-4], path=rel, dataset="nlst",
                    patient_id=pid, stratum=e.stratum, label=e.label,
                    study_yr=int(e.study_yr), timepoint=f"T{int(e.study_yr)}",
                    study_uid=s["StudyInstanceUID"], series_uid=s["SeriesInstanceUID"],
                    sop_uid=str(getattr(ds, "SOPInstanceUID", "")),
                    slice_index=idx, n_slices=len(sl),
                    series_desc=s.get("SeriesDescription", ""),
                ))
            kept_exams += 1
        except Exception as ex:
            print(f"  {pid} T{e.study_yr}: {type(ex).__name__}")
        finally:
            shutil.rmtree(dest, ignore_errors=True)
            gc.collect()

    shutil.rmtree(os.path.join(TMP, str(pid)), ignore_errors=True)

    if kept_exams >= 2:
        append(MANIFEST, rows)
        append(STATUS, [dict(pid=pid, stratum=stratum, status="ok", note=f"{kept_exams} exams")])
        ok_counts[stratum] = ok_counts.get(stratum, 0) + 1
        print(f"{pid} [{stratum}] {kept_exams} exams, {len(rows)} images "
              f"({time.time()-t0:.0f}s)  stratum {ok_counts[stratum]}/{target_per[stratum]}", flush=True)
    else:
        append(STATUS, [dict(pid=pid, stratum=stratum, status="too_few_exams", note=str(kept_exams))])
        print(f"{pid}: only {kept_exams} usable exams -- using a reserve", flush=True)
    done.add(pid)

print("\ndone for this session")

## 4. Where things stand

In [ ]:
st = pd.read_csv(STATUS)
mf = pd.read_csv(MANIFEST)
print("participants by status:"); print(st.status.value_counts().to_string())
print("\nusable participants per stratum:")
print(st[st.status=="ok"].groupby("stratum").pid.nunique().to_string())
print(f"\nimages: {len(mf):,} | participants: {mf.patient_id.nunique()} | "
      f"exams: {mf.study_uid.nunique()}")
print("\nlabels:"); print(mf.label.value_counts().to_string())
!du -sh /content/drive/MyDrive/leakage_bench/nlst